# Fetching Census Survey Data

morpc-census connects to the US Census Bureau API and returns survey data as a long-format DataFrame ready for analysis. Every query is defined by four required parameters and two optional ones:

**Required**
- `survey_table` — the dataset endpoint, e.g. `'acs/acs5'` or `'dec/pl'`
- `year` — the vintage year, e.g. `2023`
- `group` — the variable group code, e.g. `'B01001'` (sex by age)
- `scope` — the geographic extent, e.g. `'region15'` or `'franklin'`

**Optional**
- `sumlevel` — resolution within the scope, e.g. `'tract'` or `'county'`
- `variables` — a subset of variables from the group

> **Network required** — all cells below make live calls to the Census API.

In [ ]:
from morpc_census import (
    CensusAPI, DimensionTable,
    IMPLEMENTED_ENDPOINTS, get_all_avail_endpoints,
    get_table_groups, get_group_variables,
    SCOPES, PSEUDOS,
)
import pandas as pd

## 1. What surveys are available?

`IMPLEMENTED_ENDPOINTS` lists every survey/table the package supports. For any endpoint, `get_all_avail_endpoints()` returns all vintages the Census API exposes (cached after the first call).

In [ ]:
# Surveys the package supports
IMPLEMENTED_ENDPOINTS

In [ ]:
# Available vintage years for ACS 5-year
get_all_avail_endpoints()['acs/acs5']

## 2. What variables are in a group?

`get_table_groups()` returns every group in a survey with its description. `get_group_variables()` drills into a specific group to show individual variable codes and labels.

In [ ]:
# Browse available groups — show one example description
groups = get_table_groups('acs/acs5', 2023)
{k: v['description'] for k, v in list(groups.items())[:5]}

In [ ]:
# Inspect variables in group B01001 (sex by age)
vars_b01001 = get_group_variables('acs/acs5', 2023, 'B01001')
{k: v['label'] for k, v in list(vars_b01001.items())[:8]}

## 3. Choosing a scope and sumlevel

`SCOPES` names every built-in geographic extent. `PSEUDOS` shows which child sumlevels can be fetched within each parent sumlevel — the key is the parent's summary level code and the values are the allowed child codes.

In [ ]:
# Available scope names
list(SCOPES.keys())

In [ ]:
# Child sumlevels available when the scope is county-level (050)
PSEUDOS['050']

## 4. Fetching data

`CensusAPI` validates parameters, builds the request, fetches the data, and immediately transforms it into a long-format `LONG` table. The raw wide response is also stored as `DATA`.

In [ ]:
# Sex by age for all counties in the 15-county MORPC region
b01001 = CensusAPI('acs/acs5', 2023, 'B01001', scope='region15')
b01001.DATA

In [ ]:
# The long-format result — one row per geography × variable × value type
b01001.LONG

In [ ]:
# Adding sumlevel='tract' fetches all census tracts within the region's counties
b05006 = CensusAPI('acs/acs5', 2023, 'B05006', scope='region15', sumlevel='tract')
b05006.LONG

In [ ]:
# The scope and sumlevel are stored on the object for inspection
print(b05006.SCOPE, b05006.SUMLEVEL)
b05006.scope_obj    # the Scope object for this dataset's geographic extent

In [ ]:
# GEO_ID values parsed as GeoIDFQ objects — sumlevel, variant, geocomp, parts
b05006.geoidfqs[:3]

## 5. Analyzing with DimensionTable

`DimensionTable` pivots long-format data into wide and percentage tables — the same layout the Census Bureau uses in its published tables, with variable labels as a hierarchical index.

In [ ]:
# Wide format: variable labels as rows, geographies as columns
dim = DimensionTable(b01001.LONG)
dim.wide()

In [ ]:
# Percentage table — each value as a share of the Total row
dim.percent()

## 6. Time series

`DimensionTable` accepts any long-format DataFrame with the same schema. Concatenating `LONG` from multiple years produces a time-series table where `reference_period` becomes a column axis.

In [ ]:
# Fetch the same group for an earlier vintage
b01001_2018 = CensusAPI('acs/acs5', 2018, 'B01001', scope='region15')

# Stack the two years and build a dimension table
long_ts = pd.concat([b01001_2018.LONG, b01001.LONG])
DimensionTable(long_ts).wide()

## 7. Saving data

`CensusAPI.save()` writes three files to the output directory:
- `{name}.long.csv` — the long-format data
- `{name}.schema.yaml` — a frictionless Schema
- `{name}.resource.yaml` — a frictionless Resource descriptor pointing at the above two

In [ ]:
import os

b01001.save('./temp_data')

print(b01001.FILENAME)
os.path.exists(f'./temp_data/{b01001.FILENAME}')